# NB04 — Tail Risk: VaR, CVaR & Extreme Value Theory

**CRITICAL**: Cornish-Fisher uses EXCESS kurtosis with monotonicity guard.
Portfolio CVaR from joint scenarios, NOT weighted individual CVaRs.

**Output**: `var_cvar_table.csv`, `evt_parameters.csv`, `backtest_results.csv`, `return_scenarios.parquet`

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from src.config import *
from src.feature_engineering import compute_log_returns
from src.risk_metrics import *
print('Imports OK')

Imports OK


## 1. Load Data

In [2]:
master = pd.read_parquet(MASTER_DATA_FILE)
adj_tickers = [t for t in TICKERS if t in master.columns]
log_ret = compute_log_returns(master[adj_tickers])
simple_ret = master[adj_tickers].pct_change()

## 2. VaR (5 Methods × 20 Tickers)

In [3]:
var_results = []
for t in adj_tickers:
    r = log_ret[t].dropna()
    for alpha in [0.05, 0.01]:
        row = {'ticker': t, 'alpha': alpha}
        row['var_historical'] = var_historical(r, alpha)
        row['var_gaussian'] = var_parametric_gaussian(r, alpha)
        row['var_cornish_fisher'] = var_cornish_fisher(r, alpha)
        row['cvar_historical'] = cvar_historical(r, alpha)
        row['cvar_gaussian'] = cvar_parametric_gaussian(r, alpha)
        var_results.append(row)
var_df = pd.DataFrame(var_results)
var_df.to_csv(VAR_CVAR_FILE, index=False)
var_df.head(10)

Cornish-Fisher non-monotonic (z_cf=-1.4806 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.4642 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.5601 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.0342 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.5508 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.5434 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.4349 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.6004 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.5296 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.4461 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.5795 > z=-1.6449); falling back to Gaussian VaR
Cornish-Fisher non-monotonic (z_cf=-1.3738 > z=-1.6449

,ticker,alpha,var_historical,var_gaussian,var_cornish_fisher,cvar_historical,cvar_gaussian
0,NVDA,0.05,0.046323,0.049039,0.049039,0.069851,0.062040
1,NVDA,0.01,0.079618,0.070243,0.116954,0.107902,0.080787
2,AVGO,0.05,0.035427,0.038958,0.038958,0.054728,0.049171
3,AVGO,0.01,0.062928,0.055614,0.123646,0.096125,0.063897
4,TSM,0.05,0.032519,0.033841,0.033841,0.046575,0.042699
5,TSM,0.01,0.053847,0.048288,0.069022,0.071387,0.055471
6,SNPS,0.05,0.032716,0.035769,0.035769,0.051599,0.045076
7,SNPS,0.01,0.053928,0.050949,0.360346,0.089913,0.058497
8,MSFT,0.05,0.026482,0.027034,0.027034,0.039494,0.034104
9,MSFT,0.01,0.044519,0.038565,0.074213,0.063981,0.044299


## 3. EVT (POT/GPD)

In [4]:
evt_results = []
for t in adj_tickers:
    r = log_ret[t].dropna()
    gpd = fit_gpd(r, threshold_quantile=0.95)
    evt_results.append({'ticker': t, 'xi': gpd['xi'], 'beta': gpd['beta'],
        'evt_var_99': evt_var(gpd, 0.01), 'evt_cvar_99': evt_cvar(gpd, 0.01)})
evt_df = pd.DataFrame(evt_results).set_index('ticker')
evt_df.to_csv(EVT_PARAMS_FILE)
evt_df

Only 15 exceedances above threshold; GPD fit unreliable


,xi,beta,evt_var_99,evt_cvar_99
ticker,,,,
NVDA,0.087460,0.021609,0.083659,0.110918
AVGO,0.359135,0.012607,0.062887,0.097948
TSM,0.132542,0.012251,0.054492,0.071972
SNPS,0.267900,0.013187,0.059242,0.086961
MSFT,0.218946,0.010231,0.046216,0.064848
AMZN,0.055742,0.015472,0.057138,0.075061
META,0.318746,0.014780,0.065188,0.101421
GOOG,0.043181,0.014500,0.051756,0.068000
AAPL,0.149288,0.012373,0.050672,0.069165


## 4. VaR Backtesting (Kupiec + Christoffersen)

In [5]:
bt_results = []
for t in adj_tickers:
    r = log_ret[t].dropna()
    var99 = var_historical(r, 0.01)
    violations = (r < -var99).astype(int).values
    kupiec = kupiec_pof_test(violations, 0.01)
    chris = christoffersen_test(violations, 0.01)
    zone = traffic_light_zone(kupiec['violation_rate'], 0.01)
    bt_results.append({'ticker': t, 'var99': var99,
        'violation_rate': kupiec['violation_rate'],
        'kupiec_p': kupiec['p_value'], 'traffic_light': zone})
bt_df = pd.DataFrame(bt_results).set_index('ticker')
bt_df.to_csv(BACKTEST_VAR_FILE)
bt_df

,var99,violation_rate,kupiec_p,traffic_light
ticker,,,,
NVDA,0.079618,0.010313,0.874988,Green
AVGO,0.062928,0.010313,0.874988,Green
TSM,0.053847,0.010313,0.874988,Green
SNPS,0.053928,0.010313,0.874988,Green
MSFT,0.044519,0.010313,0.874988,Green
AMZN,0.056064,0.010313,0.874988,Green
META,0.063548,0.010313,0.874988,Green
GOOG,0.050369,0.010313,0.874988,Green
AAPL,0.049335,0.010313,0.874988,Green


## 5. Copula Joint Tail Risk

In [6]:
from scipy.stats import rankdata
pairs = [('NVDA', 'AMD'), ('NVDA', 'TSM'), ('CRWD', 'PANW')]
for a, b in pairs:
    if a not in adj_tickers or b not in adj_tickers: continue
    joint = pd.DataFrame({'a': log_ret[a], 'b': log_ret[b]}).dropna()
    u = rankdata(joint['a']) / (len(joint) + 1)
    v = rankdata(joint['b']) / (len(joint) + 1)
    theta = fit_clayton_copula(u, v)
    tail_dep = 2 ** (-1/theta) if theta > 0 else 0
    var_a = var_historical(joint['a'], 0.01)
    var_b = var_historical(joint['b'], 0.01)
    crash_p = joint_crash_probability(joint['a'], joint['b'], var_a, var_b)
    print(f'{a}-{b}: Clayton θ={theta:.3f}, λ_L={tail_dep:.3f}, P(joint)={crash_p:.4f}')

NVDA-AMD: Clayton θ=1.396, λ_L=0.609, P(joint)=0.0036
NVDA-TSM: Clayton θ=1.212, λ_L=0.564, P(joint)=0.0032
CRWD-PANW: Clayton θ=1.079, λ_L=0.526, P(joint)=0.0029


## 6. Return Scenario Matrix

In [7]:
scenario_matrix = build_return_scenario_matrix(simple_ret[adj_tickers])
scenario_matrix.to_parquet(RETURN_SCENARIOS_FILE)
print(f'Scenarios: {scenario_matrix.shape}, saved: {RETURN_SCENARIOS_FILE}')

Scenarios: (288, 20), saved: /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/data/features/return_scenarios.parquet


## 7. Systemic Risk Measures (CoVaR, MES, Absorption Ratio)

Cross-asset tail dependence and market-wide fragility metrics:
- **CoVaR** (Adrian & Brunnermeier, 2016): Portfolio VaR conditional on an asset being stressed
- **MES** (Acharya et al., 2017): Expected loss per asset when portfolio is in the tail
- **Absorption Ratio** (Kritzman et al., 2011): Eigenvalue-based market coupling indicator

In [8]:
from src.systemic_risk import covar_quantile_regression, mes, absorption_ratio
from src.statistical_tests import benjamini_hochberg

returns_df = log_ret[adj_tickers].dropna()

# --- Absorption Ratio (rolling 252-day, k=4) ---
ar_ts = absorption_ratio(returns_df, k=4, window=252)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(ar_ts.index, ar_ts.values, linewidth=0.8)
ax.set_title('Absorption Ratio (k=4, 252-day rolling) — Market Coupling Indicator')
ax.set_ylabel('AR')
ax.axhline(ar_ts.mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean: {ar_ts.mean():.3f}')
for ev_name, (ev_start, ev_end) in KEY_EVENTS.items():
    ax.axvspan(ev_start, ev_end, alpha=0.1, color='orange')
ax.legend()
fig.tight_layout()
from src.visualization import save_fig
save_fig(fig, 'nb04_absorption_ratio')
plt.show()

# --- MES for each ticker (equal-weight portfolio) ---
weights_ew = np.ones(len(adj_tickers)) / len(adj_tickers)
mes_values = mes(returns_df, weights_ew, alpha=0.05)
mes_df = pd.DataFrame({'ticker': adj_tickers, 'MES': mes_values}).sort_values('MES')
print("--- Marginal Expected Shortfall (Equal-Weight, α=0.05) ---")
print("Most negative MES = largest tail risk contributor")
print(mes_df.to_string(index=False))

# --- CoVaR for top-5 highest-beta names ---
port_ret = returns_df @ weights_ew
covar_results = []
high_beta_tickers = ['NVDA', 'PLTR', 'MU', 'CRWD', 'AMD']
for t in high_beta_tickers:
    if t in adj_tickers:
        result = covar_quantile_regression(port_ret, returns_df[t], alpha=0.05)
        result['ticker'] = t
        covar_results.append(result)
        print(f"\n{t}: CoVaR={result['covar']:.4f}, ΔCoVaR={result['delta_covar']:.4f}")

# --- BH-FDR on Kupiec/Christoffersen backtests ---
print("\n--- BH-FDR on VaR Backtests ---")
kupiec_pvals = bt_df['kupiec_p'].values
rejected, adjusted = benjamini_hochberg(kupiec_pvals, q=0.05)
bt_df['kupiec_p_bh'] = adjusted
bt_df['reject_bh'] = rejected
print(f"VaR models rejected (BH-corrected): {sum(rejected)}/{len(rejected)}")

# Save systemic risk outputs
systemic_df = pd.DataFrame(covar_results)
systemic_df.to_csv(TABLES_DIR / 'nb04_systemic_risk_measures.csv', index=False)
mes_df.to_csv(TABLES_DIR / 'nb04_mes_values.csv', index=False)
ar_ts.to_frame('absorption_ratio').to_csv(TABLES_DIR / 'nb04_absorption_ratio.csv')
print(f"\nSaved systemic risk measures to {TABLES_DIR}")

--- Marginal Expected Shortfall (Equal-Weight, α=0.05) ---
Most negative MES = largest tail risk contributor
ticker      MES
  AAPL 0.019649
  GOOG 0.022573
  META 0.022704
  MSFT 0.030563
  AMZN 0.032969
   CRM 0.034603
   SAP 0.039391
  PANW 0.039580
   TSM 0.042850
  SNPS 0.046739
   NOW 0.048704
   AMD 0.050815
  CRWD 0.053593
  PLTR 0.053669
  NVDA 0.054323
  AVGO 0.055169
  DDOG 0.057787
   XYZ 0.059713
  ANET 0.061552
    MU 0.067603

NVDA: CoVaR=-0.0448, ΔCoVaR=-0.0280

PLTR: CoVaR=-0.0435, ΔCoVaR=-0.0224

MU: CoVaR=-0.0443, ΔCoVaR=-0.0207

CRWD: CoVaR=-0.0410, ΔCoVaR=-0.0229

AMD: CoVaR=-0.0466, ΔCoVaR=-0.0215

--- BH-FDR on VaR Backtests ---
VaR models rejected (BH-corrected): 0/20

Saved systemic risk measures to /Users/laurentnguyen/Documents/Personal Project/Finance/Portfolio Construction/outputs/tables
